In [233]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [234]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [235]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [236]:
from sklearn.model_selection import train_test_split

#split the dataframe into 80%-training and 20%-testing
data_train,data_val =train_test_split(data,test_size =0.2 ,random_state=42)

print(data_train.shape)
print(data_val.shape)


(800, 2)
(200, 2)


## Data Preprocessing

In [237]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [238]:
import re
from bs4 import BeautifulSoup, Comment


In [239]:
# Create a copy column 
data['preprocessed_text'] = data['text']


In [240]:
import warnings
from bs4 import MarkupResemblesLocatorWarning

# 1. Turn off the BeautifulSoup URL warning so it doesn't clutter your notebook
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

# 2. Define the helper function for complete HTML structural cleaning
def clean_html_structure(text):
    # Ensure the row is treated as a string to prevent crashes on missing data
    soup = BeautifulSoup(str(text), "html.parser")
    
    # Remove all <script> and <style> tags along with their text content
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()
        
    # Correctly locate and extract hidden HTML comments row-by-row
    comments = soup.find_all(string=lambda t: isinstance(t, Comment))
    for comment in comments:
        comment.extract()
        
    # Extract only the clean text remaining between the tags
    return soup.get_text()

# 3. Apply the helper function to your train and validation splits separately
# This completely replaces the broken 'for i, text in enumerate...' loop block
data_train['preprocessed_text'] = data_train['text'].apply(clean_html_structure)
data_val['preprocessed_text'] = data_val['text'].apply(clean_html_structure)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [241]:
# Remove all the special characters ---                                                                    #pattern #empty string 
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x: re.sub(r'[^\w\s]', '', str(x)))

In [242]:
# Remove Numbers
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x: re.sub(r'\d+', '', str(x)))



In [243]:
# Remove all single characters ---
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x: re.sub(r'\s+[a-zA-Z]\s+', ' ', str(x)))


In [244]:
#  Remove single characters from the start ---
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x: re.sub(r'^[a-zA-Z]\s+', ' ', str(x)))


In [245]:
#Remove prefixed 'b' ---
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x: re.sub(r'^b\s+', '', str(x)))

In [246]:
# Substitute multiple spaces with a single space ---
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x: re.sub(r'\s+', ' ', str(x)))

In [247]:
#Convert to Lowercase ---
data['preprocessed_text'] = data['preprocessed_text'].str.lower().str.strip()

## Now let's work on removing stopwords
Remove the stopwords.

In [248]:
import nltk
from nltk.corpus import stopwords
stop_words = stopwords.words('english')

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [249]:
from nltk.stem import WordNetLemmatizer

#Initialize the Lemmatizer tool
lemmatizer = WordNetLemmatizer()

# fucntion to Lemmatization
def stop_and_lemmatize(text):
    # Split the row of text into individual words
    words = str(text).split()
    
    # Keep the word ONLY if it is not a stopword, and transform it to its base form
    cleaned_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    
    # join the remaining filtered words back together into a single text sentence
    return " ".join(cleaned_words)

In [250]:
# Apply this function across your entire DataFrame column
data['preprocessed_text'] = data['preprocessed_text'].apply(stop_and_lemmatize)


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [251]:
from collections import Counter
import pandas as pd

In [252]:
# --- 1. Filter and count words for HAM (Normal Messages) ---

# Grab all text where the label is 0 (ham) and join it into one giant string
ham_text = " ".join(data[data['label'] == 0]['preprocessed_text'].astype(str))

# Split that giant string into individual words
ham_words = ham_text.split()

# Count the frequencies of each word
ham_word_counts = Counter(ham_words)
print(ham_word_counts)

Counter({'u': 117, 'pm': 115, 'would': 106, 'state': 103, 'president': 94, 'call': 91, 'time': 84, 'percent': 77, 'secretary': 76, 'work': 73, 'mr': 71, 'one': 65, 'american': 62, 'obama': 62, 'said': 62, 'office': 60, 'ãââ': 59, 'new': 58, 'get': 57, 'year': 57, 'enough': 55, 'also': 52, 'meeting': 52, 'country': 51, 'department': 51, 'fyi': 50, 'people': 50, 'unfavorable': 48, 'know': 47, 'back': 47, 'today': 46, 'see': 46, 'issue': 44, 'day': 44, 'want': 43, 'like': 43, 'need': 42, 'house': 42, 'last': 41, 'party': 41, 'think': 39, 'could': 39, 'government': 39, 'republican': 38, 'district': 38, 'good': 37, 'first': 37, 'say': 36, 'way': 35, 'tomorrow': 34, 'right': 34, 'two': 33, 'much': 32, 'take': 32, 'minister': 32, 'let': 31, 'may': 30, 'democrat': 30, 'talk': 30, 'china': 30, 'dont': 29, 'political': 29, 'favorable': 29, 'week': 28, 'white': 28, 'policy': 27, 'public': 27, 'make': 27, 'health': 27, 'working': 27, 'heard': 27, 'official': 27, 'woman': 27, 'agreement': 27, 'arri

In [253]:
# --- 2. Filter and count words for SPAM (Fraudulent Messages) ---

# Grab all text where the label is 1 (spam) and join it into one giant string
spam_text = " ".join(data[data['label'] == 1]['preprocessed_text'].astype(str))

# Split that giant string into individual words
spam_words = spam_text.split()

# Count the frequencies of each word
spam_word_counts = Counter(spam_words)
print(spam_word_counts)

Counter({'money': 920, 'account': 793, 'bank': 744, 'fund': 702, 'u': 549, 'business': 471, 'transaction': 412, 'country': 404, 'transfer': 392, 'million': 385, 'company': 363, 'foreign': 351, 'contact': 325, 'want': 312, 'next': 309, 'sum': 292, 'dollar': 283, 'know': 281, 'name': 274, 'one': 271, 'kin': 271, 'claim': 255, 'email': 251, 'also': 251, 'mr': 250, 'security': 245, 'family': 242, 'please': 242, 'late': 230, 'government': 229, 'father': 228, 'information': 224, 'since': 223, 'year': 214, 'assistance': 206, 'state': 203, 'died': 203, 'united': 201, 'number': 195, 'come': 194, 'investment': 194, 'may': 193, 'deposit': 192, 'private': 190, 'shall': 188, 'husband': 176, 'address': 174, 'assist': 173, 'decided': 171, 'need': 171, 'would': 170, 'foreigner': 168, 'africa': 166, 'hundred': 166, 'death': 165, 'five': 163, 'contract': 159, 'made': 158, 'fax': 157, 'help': 157, 'receive': 154, 'total': 151, 'document': 151, 'make': 149, 'amount': 148, 'reply': 148, 'trust': 147, 'depo

In [254]:
print(data_train.columns)

Index(['text', 'label', 'preprocessed_text'], dtype='str')


## Extra features

In [255]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",1,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",0,0,106
535,I have not been able to reach oscar this am. W...,0,I have not been able to reach oscar this am. W...,0,0,101
695,; Huma Abedin B6I'm checking with Pat on the 5...,0,; Huma Abedin B6I'm checking with Pat on the 5...,0,0,141
557,I can have it announced here on Monday - can't...,0,I can have it announced here on Monday - can't...,0,0,52
836,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,1,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,0,1,1750


## How would work the Bag of Words with Count Vectorizer concept?

In [256]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize the vectorizer

# lowercase=True and stop_words='english' to clean up text automatically!
vectorizer = CountVectorizer(lowercase=True, stop_words='english')

#Teaching the vectorizer  vocabulary and transform the text column into numbers
X_train_bow = vectorizer.fit_transform(data_train['text'])

# Transform the validation data
X_val_bow = vectorizer.transform(data_val['text'])

# print matrix 
bow_df = pd.DataFrame(X_train_bow.toarray(), columns=vectorizer.get_feature_names_out())
bow_df.head()



,00,000,000000,00000e25,00000e251,00000eur,000066,0000ff,000m,000million,...,â½s,â½t,â½ta,â½te,â½tica,â½to,â½trangers,â½x60ã,â½x70,â½ã
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [257]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

# Load the vectorizer 
stop_words_list = list(stopwords.words('english'))
tfidf_vectorizer = TfidfVectorizer(lowercase=True, stop_words=stop_words_list)


In [258]:
#Vectorize the dataset (Fit and transform train text, only transform validation text)
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['text'])
X_val_tfidf = tfidf_vectorizer.transform(data_val['text'])

In [259]:
# Print the shape of the vectorized dataset
print("Shape of Training Vectorized Dataset:", X_train_tfidf.shape)
print("Shape of Validation Vectorized Dataset:", X_val_tfidf.shape)

Shape of Training Vectorized Dataset: (800, 23238)
Shape of Validation Vectorized Dataset: (200, 23238)


## And the Train a Classifier?

In [260]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


In [261]:
# target labels 
y_train = data_train['label']
y_val = data_val['label']

In [262]:
#Initialize the classifier (Using Naive Bayes)
classifier = MultinomialNB()

In [263]:
# Train the model using your training vectors and training labels
classifier.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[433.,367.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.61,-0.78]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 23238)","[[1.11,0. ,0. ,...,0. ,0. ,0. ], [3.7 ,6.98,0.07,...,0.03,0.03,0.35]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 23238)","[[ -9.37,-10.12,-10.12,...,-10.12,-10.12,-10.12], [ -8.65, -8.12,-10.13,...,-10.17,-10.17, -9.9 ]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,23238


In [264]:
#Predict classifications on the validation dataset
y_pred = classifier.predict(X_val_tfidf)

In [265]:
# EVALUATE PERFORMANCE
print(f"Validation Accuracy: {accuracy_score(y_val, y_pred) * 100:.2f}%\n")
print("Detailed Classification Report:")
print(classification_report(y_val, y_pred))

Validation Accuracy: 95.00%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96       125
           1       0.88      1.00      0.94        75

    accuracy                           0.95       200
   macro avg       0.94      0.96      0.95       200
weighted avg       0.96      0.95      0.95       200



### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [269]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from scipy.sparse import hstack, csr_matrix

In [270]:
# =========================================================================
# 0. PREPARE THE DATA & EXTRA FEATURES
# =========================================================================
# Extract target labels
y_train = data_train['label']
y_val = data_val['label']

# Create dense arrays for your 3 engineered features
# We wrap them in csr_matrix to make them compatible with sparse vectorizer outputs
extra_features_train = csr_matrix(data_train[['money_mark', 'suspicious_words', 'text_len']].values)
extra_features_val = csr_matrix(data_val[['money_mark', 'suspicious_words', 'text_len']].values)


In [271]:
# =========================================================================
# EXPERIMENT 1: Bag of Words Only
# =========================================================================
bow_vectorizer = CountVectorizer()
X_train_exp1 = bow_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_exp1 = bow_vectorizer.transform(data_val['preprocessed_text'])

model_exp1 = MultinomialNB() # Default parameters
model_exp1.fit(X_train_exp1, y_train)
acc_exp1 = accuracy_score(y_val, model_exp1.predict(X_val_exp1))


In [272]:
# =========================================================================
# EXPERIMENT 2: TF-IDF Only
# =========================================================================
tfidf_vectorizer = TfidfVectorizer()
X_train_exp2 = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_exp2 = tfidf_vectorizer.transform(data_val['preprocessed_text'])

model_exp2 = MultinomialNB() # Default parameters
model_exp2.fit(X_train_exp2, y_train)
acc_exp2 = accuracy_score(y_val, model_exp2.predict(X_val_exp2))

In [273]:
# =========================================================================
# EXPERIMENT 3: Bag of Words + Extra Flags
# =========================================================================
# Horizontally stack the BoW matrix with our 3 numerical feature columns
X_train_exp3 = hstack([X_train_exp1, extra_features_train])
X_val_exp3 = hstack([X_val_exp1, extra_features_val])

model_exp3 = MultinomialNB() # Default parameters
model_exp3.fit(X_train_exp3, y_train)
acc_exp3 = accuracy_score(y_val, model_exp3.predict(X_val_exp3))


In [275]:
# =========================================================================
# EXPERIMENT 4: TF-IDF + Extra Flags
# =========================================================================
# Horizontally stack the TF-IDF matrix with our 3 numerical feature columns
X_train_exp4 = hstack([X_train_exp2, extra_features_train])
X_val_exp4 = hstack([X_val_exp2, extra_features_val])

model_exp4 = MultinomialNB() # Default parameters
model_exp4.fit(X_train_exp4, y_train)
acc_exp4 = accuracy_score(y_val, model_exp4.predict(X_val_exp4))


In [276]:
# =========================================================================
# VISUALIZE THE BENCHMARK RESULTS
# =========================================================================
results_df = pd.DataFrame({
    'Feature Configuration': [
        '1. Bag of Words Only', 
        '2. TF-IDF Only', 
        '3. Bag of Words + Extra Flags', 
        '4. TF-IDF + Extra Flags'
    ],
    'Validation Accuracy': [acc_exp1, acc_exp2, acc_exp3, acc_exp4]
})

# Convert to clean percentages
results_df['Validation Accuracy'] = results_df['Validation Accuracy'].map(lambda x: f"{x*100:.2f}%")
display(results_df)

,Feature Configuration,Validation Accuracy
0,1. Bag of Words Only,92.00%
1,2. TF-IDF Only,86.00%
2,3. Bag of Words + Extra Flags,81.00%
3,4. TF-IDF + Extra Flags,57.00%
